# 07 Supervisor Agent

Run the first notebook-facing market analysis agent. The reusable agent object is created through LangChain `create_agent` in `market_analyst.services.agent`, and the notebook only sets inputs, invokes the agent, and inspects the result.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from market_analyst.config.settings import load_settings
from market_analyst.services.agent import build_market_analysis_agent
from market_analyst.telemetry import configure_notebook_logging

logger = configure_notebook_logging(run_name="07_supervisor_agent")
settings = load_settings()

settings.require_chat_model()
settings.require_database()
settings.require_embeddings()

print("Project root:", PROJECT_ROOT)
print("Chat deployment:", settings.azure_openai_chat_deployment)
print("Vector collection:", settings.vector_collection_name)

## Run Configuration

Set the sample ticker to restrict retrieval to one company, or leave it as `None` to let the tool search all indexed report chunks. The RAG store should already contain chunks from `03_rag_pipeline.ipynb` or the shared backend ingestion path.

In [ ]:
SAMPLE_TICKER = None
QUESTION = "Summarize the company's growth, debt, cash-flow, and risk signals from the annual report context."
RETRIEVAL_LIMIT = 5

if SAMPLE_TICKER:
    user_prompt = f"Ticker: {SAMPLE_TICKER}\nQuestion: {QUESTION}"
else:
    user_prompt = QUESTION

print(user_prompt)

## Create The Agent

The call below returns the LangChain agent object built with `create_agent`. The registered tool uses the shared hybrid-search module, so the agent sees the same RAG retrieval behavior that later runtime code can reuse.

In [ ]:
agent = build_market_analysis_agent(
    settings,
    retrieval_limit=RETRIEVAL_LIMIT,
)

print(type(agent))

## Invoke The Agent

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": user_prompt}]})

messages = result["messages"]
final_message = messages[-1]
final_text = getattr(final_message, "content", final_message)
print(final_text)

## Inspect Tool Trace

This cell prints a compact trace so you can confirm the model actually used the retrieval tool when report context is needed.

In [ ]:
for index, message in enumerate(messages, start=1):
    message_type = getattr(message, "type", type(message).__name__)
    tool_calls = getattr(message, "tool_calls", None)
    print(f"{index}. {message_type}")
    if tool_calls:
        print("   tool_calls:", tool_calls)
    content = getattr(message, "content", "")
    if content:
        preview = str(content).replace("\n", " ")[:240]
        print("   content:", preview)

## Validation

In [ ]:
assert messages, "Agent result should include messages."
assert str(final_text).strip(), "Agent final response should not be empty."

print("Agent notebook validation passed.")